# Experiment 2: Crossover Analysis

Fix Read-Heavy (80/20) profile and independently vary three factors:
- **Update intensity** ($M_U$): how much of the table is updated per TX
- **History-scan share** (HS): fraction of scans that reuse old snapshots
- **Delta-scan frequency** ($f_\Delta$): fraction of reads that are delta scans

**Visualization**: Absolute latency (ms/tx) with SNAP, IVMH, and best MVHT configs as separate lines.
Crossover = where MVHT line drops below baseline line.

In [ ]:
import subprocess, re, os
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

# ── Config ──────────────────────────────────────────────────────────────
BIN = Path("../../../target/release/htap_wkld")
OUT_DIR = Path("output_exp2")
OUT_DIR.mkdir(exist_ok=True)

TABLE_TYPES = ["naive", "ivmh", "heap", "par", "chain"]
BASELINES = {"naive", "ivmh"}

REPEAT = 3          # ← 여기서 반복 횟수 조절
TXN_NUM = 100
TXN_GC_RATIO = 0.05

BASE_ARGS = [
    "--txn-count", "60",
    "--update-ratio", "0.003",
    "--warehouse-count", "10",
    "--probe-ratio", "0.0001",
]

## 1. Build

In [ ]:
# ── Build ──
r = subprocess.run(["cargo", "build", "--release", "--bin", "htap_wkld"],
                   capture_output=True, text=True)
if r.returncode != 0:
    print(r.stderr)
    raise SystemExit(1)
print("Build OK")

# ── Quick smoke test: tiny params, 10s timeout ──
smoke_cmd = [
    str(BIN), "--txn-count", "5", "--warehouse-count", "1",
    "--update-ratio", "0.003", "--probe-ratio", "0.0001",
    "--txn-update-ratio", "0.2", "--txn-delta-ratio", "0.0",
    "--txn-probe-ratio", "0.4", "--txn-scan-ratio", "0.4",
    "--txn-gc-ratio", "0.05", "--table-type", "naive",
]
try:
    sr = subprocess.run(smoke_cmd, capture_output=True, text=True, timeout=10)
    print(f"Smoke test: rc={sr.returncode}, stdout lines={len(sr.stdout.splitlines())}")
    if sr.returncode != 0:
        print("STDERR:", sr.stderr[:500])
    else:
        # Show last 10 lines of output
        for line in sr.stdout.splitlines()[-10:]:
            print(" ", line)
except subprocess.TimeoutExpired:
    print("Smoke test TIMEOUT (>10s) — binary likely hangs")

## 2. Run & Collect

In [ ]:
from bench_script_functions import parse_result

TX_MAP = {"MarkTs": "BuildSnap", "DelSc": "DeltaScan"}

TIMEOUT_SEC = 120   # ← per-subprocess timeout (seconds)


def merge_args(base, extra):
    """Merge CLI arg lists; extra overrides base when the same flag appears."""
    merged = {}
    for lst in (base, extra):
        it = iter(lst)
        for tok in it:
            if tok.startswith("--"):
                merged[tok] = next(it)
            else:
                merged[tok] = None  # positional (shouldn't happen)
    result = []
    for k, v in merged.items():
        result.append(k)
        if v is not None:
            result.append(v)
    return result


def run_single(extra_args):
    """Run all table types with given args, return per-(table, repair) total latency."""
    rows = []
    args = merge_args(BASE_ARGS, extra_args)
    for tt in TABLE_TYPES:
        print(f"[{tt}]", end="", flush=True)
        cmd = [str(BIN)] + args + ["--table-type", tt]
        durations = []
        skip = False
        for rep in range(REPEAT):
            try:
                r = subprocess.run(cmd, capture_output=True, text=True,
                                   timeout=TIMEOUT_SEC)
            except subprocess.TimeoutExpired:
                print(f" TIMEOUT(rep={rep})", end="", flush=True)
                skip = True
                break
            if r.returncode != 0:
                print(f" ERR(rc={r.returncode})", end="", flush=True)
                if r.stderr:
                    print(f"\n  stderr: {r.stderr[:500]}", flush=True)
                skip = True
                break
            df = parse_result(r.stdout, tt)
            df["tx_type"] = df["tx_type"].replace(TX_MAP)
            total = df.groupby("repair_type")["duration_ms"].sum()
            durations.append(total)
        if skip or len(durations) == 0:
            print(" SKIP ", end="", flush=True)
            continue
        avg = pd.concat(durations, axis=1).mean(axis=1)
        for repair, dur in avg.items():
            if tt in BASELINES:
                if repair != "Write Repair":
                    continue
                rows.append({"table_type": tt, "repair_type": "",
                             "total_ms": dur / TXN_NUM})
            else:
                rows.append({"table_type": tt, "repair_type": repair,
                             "total_ms": dur / TXN_NUM})
        print(" ok", end=" ", flush=True)
    print()
    return pd.DataFrame(rows)


def sweep_factor(factor_name, factor_values, base_analytical=0.80,
                 make_args_fn=None):
    """Sweep one factor, collecting absolute latency for all configs."""
    all_rows = []
    for val in factor_values:
        print(f"  {factor_name}={val:.3f} ", end="", flush=True)
        extra = make_args_fn(val, base_analytical)
        df = run_single(extra)
        df[factor_name] = val
        all_rows.append(df)
        print("  done")
    return pd.concat(all_rows, ignore_index=True)

In [ ]:
# ── Factor 1: Delta-scan frequency ────────────────────────────────────
def delta_args(delta_ratio, analytical):
    update = 1.0 - analytical
    remaining = analytical - delta_ratio
    probe = remaining / 2
    scan = remaining / 2
    return [
        "--txn-update-ratio", str(update),
        "--txn-delta-ratio", str(delta_ratio),
        "--txn-probe-ratio", str(probe),
        "--txn-scan-ratio", str(scan),
        "--txn-gc-ratio", str(TXN_GC_RATIO),
    ]

DELTA_VALUES = [0.00, 0.05, 0.10, 0.15, 0.20, 0.25]

csv_path = OUT_DIR / "exp2_delta.csv"
if csv_path.exists():
    print("exp2_delta.csv exists, skipping run")
    df_delta = pd.read_csv(csv_path, keep_default_na=False)
else:
    print("=== Delta-scan sweep ===")
    df_delta = sweep_factor("delta_ratio", DELTA_VALUES, make_args_fn=delta_args)
    df_delta.to_csv(csv_path, index=False)
    print("Saved.")

In [ ]:
# ── Factor 2: Update intensity ────────────────────────────────────────
UPDATE_RATIOS = [0.01, 0.05, 0.10, 0.15, 0.20]

def update_args(update_intensity, analytical):
    """update_intensity = fraction of base table updated per TX."""
    return [
        "--analytical-ratio", str(analytical),
        "--txn-gc-ratio", str(TXN_GC_RATIO),
        "--update-ratio", str(update_intensity),
    ]

csv_path = OUT_DIR / "exp2_update.csv"
if csv_path.exists():
    print("exp2_update.csv exists, skipping run")
    df_update = pd.read_csv(csv_path, keep_default_na=False)
else:
    print("=== Update-intensity sweep ===")
    df_update = sweep_factor("update_ratio", UPDATE_RATIOS, make_args_fn=update_args)
    df_update.to_csv(csv_path, index=False)
    print("Saved.")

In [ ]:
# ── Factor 3: History-scan share (scan_reuse_ratio) ───────────────────
REUSE_VALUES = [0.0, 0.2, 0.4, 0.6, 0.8, 1.0]

def reuse_args(reuse_ratio, analytical):
    return [
        "--analytical-ratio", str(analytical),
        "--txn-gc-ratio", str(TXN_GC_RATIO),
        "--scan-reuse-ratio", str(reuse_ratio),
    ]

csv_path = OUT_DIR / "exp2_reuse.csv"
if csv_path.exists():
    print("exp2_reuse.csv exists, skipping run")
    df_reuse = pd.read_csv(csv_path, keep_default_na=False)
else:
    print("=== History-scan share sweep ===")
    df_reuse = sweep_factor("reuse_ratio", REUSE_VALUES, make_args_fn=reuse_args)
    df_reuse.to_csv(csv_path, index=False)
    print("Saved.")

## 3. Plot — Absolute Latency with Two Baselines

In [ ]:
# ── Plotting ───────────────────────────────────────────────────────────
TOL = {
    "blue": "#4477AA", "cyan": "#66CCEE", "green": "#228833",
    "yellow": "#CCBB44", "red": "#EE6677", "purple": "#AA3377",
    "grey": "#BBBBBB",
}

# Color by table type
TYPE_COLOR = {
    "naive": TOL["grey"],   # SNAP
    "ivmh":  TOL["yellow"], # IVMH
    "heap":  TOL["blue"],   # MONO
    "chain": TOL["green"],  # DUAL
    "par":   TOL["red"],    # EPOCH
}
# Linestyle by repair
REPAIR_LS = {"No Repair": "--", "Read Repair": ":", "Write Repair": "-", "": "-"}
# Markers
REPAIR_MK = {"No Repair": "^", "Read Repair": "v", "Write Repair": "o", "": "s"}

TABLE_DISPLAY = {"naive":"SNAP", "ivmh":"IVMH", "heap":"MONO", "chain":"DUAL", "par":"EPOCH"}
REPAIR_DISPLAY = {"No Repair":"NR", "Read Repair":"RR", "Write Repair":"WR", "":""}

plt.rcParams["font.family"] = "serif"
plt.rcParams["font.serif"] = ["Times New Roman","Times","DejaVu Serif"]


def plot_crossover(df, x_col, xlabel, title, ylim=None):
    """Line plot: absolute latency vs factor, one line per (table, repair)."""
    # Deduplicate baselines
    df = df.drop_duplicates(subset=[x_col, "table_type", "repair_type"])
    
    fig, ax = plt.subplots(figsize=(7, 5))
    
    # Plot order: baselines first (thicker), then MVHT configs
    groups = df.groupby(["table_type", "repair_type"])
    
    # Baselines
    for (tt, rp), gdf in groups:
        if tt not in ["naive", "ivmh"]:
            continue
        gdf = gdf.sort_values(x_col)
        lbl = TABLE_DISPLAY[tt]
        ax.plot(gdf[x_col], gdf["total_ms"],
                color=TYPE_COLOR[tt], linestyle="-", linewidth=2.5,
                marker="s", markersize=6, label=lbl, zorder=5)
    
    # MVHT configs
    for (tt, rp), gdf in groups:
        if tt in ["naive", "ivmh"]:
            continue
        # Skip DUAL non-WR and CHAIN non-WR
        if tt == "chain" and rp != "Write Repair":
            continue
        gdf = gdf.sort_values(x_col)
        rp_short = REPAIR_DISPLAY.get(rp, rp)
        lbl = f"{TABLE_DISPLAY[tt]}-{rp_short}"
        ax.plot(gdf[x_col], gdf["total_ms"],
                color=TYPE_COLOR[tt],
                linestyle=REPAIR_LS.get(rp, "-"),
                linewidth=1.8,
                marker=REPAIR_MK.get(rp, "o"),
                markersize=4, label=lbl)
    
    ax.set_xlabel(xlabel, fontsize=11)
    ax.set_ylabel("Latency (ms / tx)", fontsize=11)
    ax.grid(True, linestyle="--", linewidth=0.5, alpha=0.6)
    if ylim:
        ax.set_ylim(ylim)
    ax.legend(fontsize=8.5, ncol=2, framealpha=0.9, loc="best")
    plt.tight_layout()
    plt.savefig(OUT_DIR / f"{title}.pdf", format="pdf")
    plt.show()

In [ ]:
# ── Load & plot ────────────────────────────────────────────────────────
for csv_name, x_col, xlabel, title, df_var in [
    ("exp2_delta.csv",  "delta_ratio",  "Delta-Scan Portion ($f_\\Delta$)",  "exp2_delta",  "df_delta"),
    ("exp2_update.csv", "update_ratio", "Update Intensity ($M_U$)",          "exp2_update", "df_update"),
    ("exp2_reuse.csv",  "reuse_ratio",  "History-Scan Share (HS)",           "exp2_reuse",  "df_reuse"),
]:
    csv_path = OUT_DIR / csv_name
    if csv_path.exists():
        df = pd.read_csv(csv_path, keep_default_na=False)
    else:
        df = locals().get(df_var)
    if df is None:
        print(f"Skipping {csv_name} — not found")
        continue
    df["repair_type"] = df["repair_type"].fillna("")  # safety net
    print(f"\n=== {title} ===")
    plot_crossover(df, x_col, xlabel, title)